<a href="https://colab.research.google.com/github/faizanarif2/PyTorch_MNIST_NeuralNetwork/blob/main/MNIST_PyTorchLightning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

In [16]:
class DataManagement(pl.LightningModule): #Making separate class for data management
  def __init__(self,data_dir:str="./data",batch_size: int=32):
    super().__init__()
    self.data_dir=data_dir
    self.batch_size=batch_size
    self.transform=transforms.ToTensor()

  def prepare_data(self):
    MNIST(self.data_dir,train=True,download=True)
    MNIST(self.data_dir,train=False,download=True)

  def setup(self, stage=None):
    self.mnist_train=MNIST(self.data_dir,train=True,transform=self.transform)
    self.mnist_val=MNIST(self.data_dir,train=False,transform=self.transform)

  def train_dataloader(self):
    return DataLoader(self.mnist_train,batch_size=self.batch_size,shuffle=True)

  def val_dataloader(self):
    return DataLoader(self.mnist_val,batch_size=self.batch_size)



In [20]:
from pytorch_lightning.callbacks.progress import progress_bar
class Recognizer(pl.LightningModule):

  def __init__(self):
    super().__init__()
    self.model=nn.Sequential(
        nn.Flatten(),
        nn.Linear(784,128),
        nn.ReLU(),
        nn.Linear(128,10)
    )
    self.loss_fn=nn.CrossEntropyLoss()

  def forward(self,x):
    return self.model(x)

  def training_step(self,batch,batch_idx):
    batch_items,batch_labels=batch
    predictions=self(batch_items)
    loss=self.loss_fn(predictions,batch_labels)
    self.log("train_loss",loss,prog_bar=True,on_step=False,on_epoch=True)
    return loss

  def validation_step(self,batch,batch_idx):
    batch_items,batch_labels=batch
    predictions=self(batch_items)
    validation_loss=self.loss_fn(predictions,batch_labels)
    accuracy=(predictions.argmax(dim=1)==batch_labels).float().mean()
    self.log("val_loss",validation_loss,prog_bar=True)
    self.log("val_acc",accuracy,prog_bar=True)
    return validation_loss

  def configure_optimizers(self):
    return torch.optim.Adam(self.parameters(),lr=0.001)



In [21]:
dataModule=DataManagement()
model=Recognizer()

checkpoint_callback=ModelCheckpoint(
    monitor="val_loss",
    mode="min",
    save_top_k=1,
    filename="new-mnist-model"
)

early_stopping_callback=EarlyStopping(
    monitor="val_loss",
    patience=3,
    mode="min"
)

trainer=pl.Trainer(
    max_epochs=5,
    callbacks=[checkpoint_callback,early_stopping_callback],
    accelerator="auto"
)

trainer.fit(model, datamodule=dataModule)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ Sequential       │  101 K │ train │     0 │
│ 1 │ loss_fn │ CrossEntropyLoss │      0 │ train │     0 │
└───┴─────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 101 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 101 K                                                                                                
Total estimated model params size (MB): 0.407                                                                      
Modules in train mode: 6                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.13/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=5` reached.
